In [ ]:
import numpy as np
import pandas as pd
import os
import json
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
ARTIFACTS_DIR =  Path.cwd().parent.joinpath("artifacts")

In [ ]:
embeddings = np.load(os.path.join(ARTIFACTS_DIR, "embeddings.npy"))
metadata = pd.read_parquet(os.path.join(ARTIFACTS_DIR, "metadata.parquet"))
config = json.load(open(os.path.join(ARTIFACTS_DIR, "embedding_config.json"), "r"))

print("Embeddings: ", embeddings.shape)
print("Metadata: ", metadata.shape)
print("Configuration: ", config)

In [ ]:
model = SentenceTransformer(config["embedding_model"], device = "cuda" if torch.cuda.is_available() else "cpu")
print("Model:", model)
print("Device:", model.device)
print("Embedding dimension:", model.get_embedding_dimension())

In [ ]:
def retrieve(query, top_k = 5):
    query = model.encode(query)

    scores = cosine_similarity(query.reshape(1, -1), embeddings)[0]

    top_k = np.argsort(scores)[::-1][:top_k]

    results = metadata.iloc[top_k].copy()
    results["similarity"] = scores[top_k]

    return results.reset_index(drop = True)

In [ ]:
query = "What is the chemical class of Allegra?"

results = retrieve(query, top_k = 3)

display(results)

In [ ]:
def search(query, top_k=5):

    results = retrieve(query, top_k = top_k)

    for i, row in results.iterrows():

        print("=" * 80)
        print(f"RESULT #{i + 1}")
        print(f"Similarity: {row['similarity']:.4f}")
        print()

        print(row["document"])

In [ ]:
test_queries = [
    "What are alternatives to paracetamol?",
    "What are the side effects of ibuprofen?",
    "Which medicines are used for pain relief?",
    "What drugs are similar to aspirin?",
    "Which medicines can cause nausea?"
]

for query in test_queries:

    print("\n\n")
    print("#" * 100)
    print("QUERY:", query)

    search(query, top_k=3)